In [ ]:
# %%
# abc1.ipynb - cell 1: setup, imports, configuration (forced ground truth labels)

import pandas as pd
import numpy as np
from pathlib import Path
import sys
import random
import joblib
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             roc_curve, precision_recall_curve)
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Optional, Any, Tuple, Dict

# --- project root ---
# sets up the project's root path for imports
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# --- import from our csi_preprocessing module ---
# brings in helper functions for loading config and data paths
from csi_preprocessing import (
    load_app_config,
    get_raw_data_path,
    get_config_csi_metadata_cols,
    remove_known_dead_csi_columns,
    ensure_plot_dir,
)
print("necessary functions from csi_preprocessing imported.")

# --- load global configuration ---
# loads the main config.json file
print("\n--- loading global configuration ---")
try:
    config = load_app_config(PROJECT_ROOT)
    print("global configuration loaded successfully.")
    METADATA_COLS_FROM_CONFIG = get_config_csi_metadata_cols(config)
    # defines metadata columns to be dropped during cleaning for this specific pipeline
    OLD_PIPELINE_COLUMNS_TO_REMOVE = ["conf", "det ts", "csi ts"]
    TARGET_LABEL_COL = config.get("target_label_column", "people")
except Exception as e:
    print(f"error loading global configuration: {e}")
    raise

# --- experiment-specific configuration for abc1.ipynb (forced ground truth) ---
# sets a unique name for this experiment based on the config file
EXPERIMENT_NAME = "Forced_GT_Labels_Old_Pipeline_Rep"
PIPELINE_CONFIG_KEY = "ground_truth_labels"

if PIPELINE_CONFIG_KEY not in config.get("experiment_pipelines", {}):
    raise ValueError(f"pipeline configuration for '{PIPELINE_CONFIG_KEY}' not found in config.json")

EXPERIMENT_SUFFIX = config["experiment_pipelines"][PIPELINE_CONFIG_KEY]["pipeline_suffix"]

# --- define output paths for this experiment ---
# creates all necessary directories for storing outputs of this fgt notebook
BASE_OUTPUT_DIR_ABC1 = PROJECT_ROOT / f"experiment_outputs_{EXPERIMENT_SUFFIX}"
ABC1_CLEAN_DIR = BASE_OUTPUT_DIR_ABC1 / "data_clean"
ABC1_ARTIFACTS_DIR = BASE_OUTPUT_DIR_ABC1 / "artifacts" # for scaler, pca model
ABC1_WINDOWED_DATA_DIR = BASE_OUTPUT_DIR_ABC1 / "windowed_data" # for final x,y arrays
ABC1_TRAIN_WINDOW_DIR = ABC1_WINDOWED_DATA_DIR / "train"
ABC1_VAL_WINDOW_DIR = ABC1_WINDOWED_DATA_DIR / "val"
ABC1_TEST_WINDOW_DIR = ABC1_WINDOWED_DATA_DIR / "test"
ABC1_MODELS_DIR = PROJECT_ROOT / config["models_storage_base_dir"] / f"csi_detector_{EXPERIMENT_SUFFIX}"
ABC1_PLOTS_DIR = PROJECT_ROOT / config["plots_base_dir"] / f"training_plots_{EXPERIMENT_SUFFIX}"

for dir_path in [ABC1_CLEAN_DIR, ABC1_ARTIFACTS_DIR, ABC1_WINDOWED_DATA_DIR,
                 ABC1_TRAIN_WINDOW_DIR, ABC1_VAL_WINDOW_DIR, ABC1_TEST_WINDOW_DIR,
                 ABC1_MODELS_DIR, ABC1_PLOTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"\n--- paths for experiment: {EXPERIMENT_NAME} ---")
print(f"clean data dir: {ABC1_CLEAN_DIR}")
print(f"artifacts (scaler, pca) dir: {ABC1_ARTIFACTS_DIR}")
print(f"windowed data dir: {ABC1_WINDOWED_DATA_DIR}")
print(f"trained models dir: {ABC1_MODELS_DIR}")
print(f"training plots dir: {ABC1_PLOTS_DIR}")

# --- replication pipeline parameters ---
# sets random seed for reproducibility across libraries
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ensures dwt is applied before pca, as described in the paper
DWT_BEFORE_PCA = True

# sets parameters for dwt and pca, matching the paper's methods section
DWT_WAVELET_OLD = 'db4'
DWT_LEVEL_OLD = 4
PCA_N_COMPONENTS_OLD = 0.99 # retain 99% of variance
PCA_SVD_SOLVER_OLD = "full"
PCA_RANDOM_STATE_OLD = SEED

# sets parameters for the temporal windowing step
WIN_SIZE_ABC1 = 16
STEP_SIZE_ABC1 = 8

# sets parameters for the mlp model training
MLP_EPOCHS_ABC1 = 60
MLP_BATCH_SIZE_ABC1 = 512
MLP_LEARNING_RATE_ABC1 = 3e-4
PLOT_DPI = 200
sns.set_theme(style="whitegrid", palette="pastel")

print("\n✔ cell 1: configuration for abc1.ipynb (forced gt labels) done.")
print(f"dwt_before_pca set to: {DWT_BEFORE_PCA}")

In [ ]:
# %%
# abc1.ipynb - cell 2: initial data cleaning (forced gt labels)

print(f"\n--- starting data cleaning for: {EXPERIMENT_NAME} ---")
files_cleaned_count_abc1 = 0
for category_key, sub_categories_map in config["raw_data_categories"].items():
    # assigns the ground truth label (0 or 1) based on the raw data folder name
    ground_truth_label = -1
    if "0_pers" in category_key.lower(): ground_truth_label = 0
    elif "1_pers" in category_key.lower(): ground_truth_label = 1

    if ground_truth_label == -1:
        print(f"warning: could not assign ground truth label for category '{category_key}'. skipping.")
        continue

    for sub_category_key, _ in sub_categories_map.items():
        current_raw_data_dir = get_raw_data_path(PROJECT_ROOT, config, category_key, sub_category_key)
        output_clean_subdir_abc1 = ABC1_CLEAN_DIR / category_key / sub_category_key
        output_clean_subdir_abc1.mkdir(parents=True, exist_ok=True)

        if not current_raw_data_dir.exists():
            print(f"warning: raw data directory not found: {current_raw_data_dir}")
            continue

        # iterates through each raw csv file for cleaning
        for raw_csv_file_path in sorted(list(current_raw_data_dir.glob('*.csv'))):
            try:
                df_raw = pd.read_csv(raw_csv_file_path)

                # 1. drops unused metadata columns like confidence and timestamps
                df_processing = df_raw.drop(columns=[c for c in OLD_PIPELINE_COLUMNS_TO_REMOVE if c in df_raw.columns], errors='ignore')

                # 2. forces the label column to the known ground truth value
                df_processing[TARGET_LABEL_COL] = ground_truth_label

                # 3. removes the 12 inactive (dead) subcarriers, resulting in 104 csi features
                df_processing = remove_known_dead_csi_columns(df_processing.copy())

                # saves the cleaned dataframe to a new file
                output_file_path = output_clean_subdir_abc1 / raw_csv_file_path.name
                df_processing.to_csv(output_file_path, index=False)
                files_cleaned_count_abc1 += 1
            except Exception as e:
                print(f"error cleaning file {raw_csv_file_path.name}: {e}")
                import traceback; traceback.print_exc()

print(f"\n✔ cell 2: data cleaning for {EXPERIMENT_NAME} complete. {files_cleaned_count_abc1} files cleaned into '{ABC1_CLEAN_DIR}'.")

In [ ]:
# %%
# abc1.ipynb - cell 3: define "old pipeline" processing helper functions

# loads a cleaned csv file into numpy arrays for features (x) and labels (y)
def load_cleaned_abc1_csv(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(path)
    if TARGET_LABEL_COL not in df.columns:
        raise ValueError(f"target label column '{TARGET_LABEL_COL}' not found in {path}")
    X = df.drop(columns=[TARGET_LABEL_COL], errors='ignore').values.astype(np.float32)
    y = df[TARGET_LABEL_COL].values.astype(np.int64)
    return X, y

# applies dwt soft-threshold denoising to each feature column of the data matrix
def dwt_denoise_matrix_abc1(X: np.ndarray, wavelet: str = DWT_WAVELET_OLD, level: int = DWT_LEVEL_OLD) -> np.ndarray:
    n_rows, n_features = X.shape
    Xd = np.zeros_like(X)
    if n_rows == 0: return Xd
    wavelet_obj = pywt.Wavelet(wavelet)
    for i in range(n_features):
        col = X[:, i]
        current_max_level = pywt.dwt_max_level(len(col), wavelet_obj)
        actual_level = min(level, current_max_level)
        if actual_level < 1: Xd[:, i] = col; continue
        coeffs = pywt.wavedec(col, wavelet, level=actual_level)
        detail_coeffs_finest = coeffs[-1]
        sigma = 0
        if len(detail_coeffs_finest) > 0: sigma = np.median(np.abs(detail_coeffs_finest - np.median(detail_coeffs_finest))) / 0.6745
        thresh = 0.0
        if sigma > 0 and n_rows > 1: thresh = sigma * np.sqrt(2 * np.log(n_rows))
        elif sigma > 0 and n_rows == 1: thresh = sigma
        coeffs_thresholded = [coeffs[0]] + [pywt.threshold(c, thresh, mode="soft") for c in coeffs[1:]]
        reconstructed_col = pywt.waverec(coeffs_thresholded, wavelet)
        if len(reconstructed_col) > n_rows: Xd[:, i] = reconstructed_col[:n_rows]
        elif len(reconstructed_col) < n_rows and len(reconstructed_col) > 0:
            Xd[:, i] = np.concatenate([reconstructed_col, np.full(n_rows - len(reconstructed_col), reconstructed_col[-1])])
        elif len(reconstructed_col) == 0 and n_rows > 0: Xd[:, i] = np.zeros(n_rows)
        else: Xd[:, i] = reconstructed_col
    return Xd

# applies a previously fitted pca model to transform data
def apply_pca_abc1(X: np.ndarray, pca_model: PCA) -> np.ndarray:
    if not hasattr(pca_model, 'n_components_') or pca_model.n_components_ == 0: return X
    if X.shape[1] != pca_model.n_features_in_:
        print(f"pca warning: input features ({X.shape[1]}) != pca fit features ({pca_model.n_features_in_}). returning original.")
        return X
    return pca_model.transform(X)

# creates overlapping 3d windows from the 2d time-series data
# the label for each window is determined by a majority vote of the frames within it
def window_data_abc1_3d(X: np.ndarray, y: np.ndarray, window_size: int, step_size: int) -> Tuple[np.ndarray, np.ndarray]:
    segments, labels = [], []
    N, num_features = X.shape
    if N < window_size or num_features == 0: return np.array([]).reshape(0, window_size, num_features), np.array([])
    for start in range(0, N - window_size + 1, step_size):
        segments.append(X[start : start + window_size])
        label_segment = y[start : start + window_size]
        labels.append(int(np.mean(label_segment) > 0.5) if len(label_segment) > 0 else 0)
    if not segments: return np.array([]).reshape(0, window_size, num_features), np.array([])
    return np.stack(segments).astype(np.float32), np.array(labels).astype(np.int64)

print("✔ cell 3: helper functions for abc1.ipynb defined.")

In [ ]:
# %%
# abc1.ipynb - cell 4: stratified file splitting, scaler & pca fitting

print(f"\n--- starting stratified file splitting, scaler & pca fitting for: {EXPERIMENT_NAME} ---")

# 1. gathers all cleaned csv files and groups them by their sub-location
files_by_sublocation_abc1 = {}
for category_key, sub_categories_map in config["raw_data_categories"].items():
    for sub_category_key, _ in sub_categories_map.items():
        sub_loc_key = f"{category_key}_{sub_category_key}"
        current_clean_sub_dir = ABC1_CLEAN_DIR / category_key / sub_category_key
        if current_clean_sub_dir.exists():
            files = sorted(list(current_clean_sub_dir.glob("*.csv")))
            if files: files_by_sublocation_abc1[sub_loc_key] = files
print("\nfiles collected per sub-location for abc1:")
for sub_loc, f_list in files_by_sublocation_abc1.items(): print(f"  {sub_loc}: {len(f_list)} files")

# 2. performs a stratified split on the *files*, not the data rows
# this ensures that data from all locations is present in both train and test sets
train_files_abc1, test_files_abc1 = [], []
np.random.seed(SEED)
for sub_loc_key, files_in_sub_loc in files_by_sublocation_abc1.items():
    if not files_in_sub_loc: continue
    n_test_sub_loc = 0
    if len(files_in_sub_loc) > 1 :
        n_test_sub_loc = max(1, int(len(files_in_sub_loc) * 0.20))
        if len(files_in_sub_loc) - n_test_sub_loc < 1:
             n_test_sub_loc = len(files_in_sub_loc) - 1
    if n_test_sub_loc > 0 :
        chosen_test = list(np.random.choice(files_in_sub_loc, n_test_sub_loc, replace=False))
        chosen_train = [f for f in files_in_sub_loc if f not in chosen_test]
        test_files_abc1.extend(chosen_test)
        train_files_abc1.extend(chosen_train)
    else:
        train_files_abc1.extend(files_in_sub_loc)

print(f"\ntotal cleaned files for abc1: {sum(len(v) for v in files_by_sublocation_abc1.values())}")
print(f"abc1 train files: {len(train_files_abc1)}")
print(f"abc1 test files: {len(test_files_abc1)}")

# 3. stacks data from *only* the training files to create a fitting dataset
# this is critical to prevent data leakage from the test set
X_stack_abc1_list, y_stack_abc1_list_gt = [], []
for p_path in train_files_abc1:
    X_f, y_f = load_cleaned_abc1_csv(p_path)
    if X_f.shape[0] > 0: X_stack_abc1_list.append(X_f); y_stack_abc1_list_gt.append(y_f)
if not X_stack_abc1_list: raise ValueError("no data in training files for abc1.")
X_stack_abc1 = np.vstack(X_stack_abc1_list)

print(f"abc1 x_stack shape (for scaler/pca): {X_stack_abc1.shape}")

# 4. fits the standardscaler on the training data stack and saves it
scaler_abc1 = StandardScaler().fit(X_stack_abc1)
scaler_save_path_abc1 = ABC1_ARTIFACTS_DIR / "scaler_abc1.joblib"
joblib.dump(scaler_abc1, scaler_save_path_abc1)
print(f"abc1 scaler saved: {scaler_save_path_abc1}")

# 5. scales the data, applies dwt, then fits the pca model and saves it
Xn_abc1 = scaler_abc1.transform(X_stack_abc1)
X_for_pca_fit_abc1 = Xn_abc1
if DWT_BEFORE_PCA:
    print("applying dwt before pca to abc1 training stack...")
    X_for_pca_fit_abc1 = dwt_denoise_matrix_abc1(Xn_abc1.copy())
else:
    print("skipping dwt before pca for abc1 (dwt applied later to pcs per file).")

if X_for_pca_fit_abc1.shape[1] == 0: raise ValueError("no features for pca in abc1.")
pca_abc1 = PCA(n_components=PCA_N_COMPONENTS_OLD, svd_solver=PCA_SVD_SOLVER_OLD, random_state=PCA_RANDOM_STATE_OLD)
pca_abc1.fit(X_for_pca_fit_abc1)
pca_save_path_abc1 = ABC1_ARTIFACTS_DIR / "pca_abc1.joblib"
joblib.dump(pca_abc1, pca_save_path_abc1)
print(f"abc1 pca model saved: {pca_save_path_abc1}")
n_pca_comp = getattr(pca_abc1, 'n_components_', 0)
print(f"  pca components fitted: {n_pca_comp}")
if n_pca_comp > 0: print(f"  explained variance: {np.sum(pca_abc1.explained_variance_ratio_):.4f}")

print(f"\n✔ cell 4: scaler and pca for {EXPERIMENT_NAME} fitted and saved.")

In [ ]:
# %%
# abc1.ipynb - cell 5: windowing and processing for each split

print(f"\n--- starting windowing for: {EXPERIMENT_NAME} ---")

# reloads the fitted scaler and pca model artifacts
scaler_loaded_abc1 = joblib.load(ABC1_ARTIFACTS_DIR / "scaler_abc1.joblib")
pca_loaded_abc1 = joblib.load(ABC1_ARTIFACTS_DIR / "pca_abc1.joblib")
print("reloaded abc1 scaler and pca model.")
print(f"using fixed win_size_abc1={WIN_SIZE_ABC1}, step_size_abc1={STEP_SIZE_ABC1}")

apply_dwt_after_pca_flag_abc1 = not DWT_BEFORE_PCA

# this function applies the full processing pipeline to a list of files
# it uses the *pre-fitted* scaler and pca models
def process_data_split_abc1(files_list: List[Path],
                            scaler: StandardScaler, pca_model: PCA,
                            win_sz: int, step_sz: int,
                            dwt_after_pca: bool, dwt_before_pca: bool):
    print(f"processing {len(files_list)} files for windowing...")
    X_all_win, y_all_win = [], []
    for p_path in files_list:
        X_file, y_file_gt = load_cleaned_abc1_csv(p_path)
        if X_file.shape[0] < win_sz: continue
        # the full pipeline: scale -> dwt -> pca -> window
        Xn = scaler.transform(X_file)
        X_pre_pca = Xn
        if dwt_before_pca: X_pre_pca = dwt_denoise_matrix_abc1(Xn.copy())
        Xp = apply_pca_abc1(X_pre_pca, pca_model)
        X_final = Xp
        if dwt_after_pca: X_final = dwt_denoise_matrix_abc1(Xp.copy())
        Xw, yw = window_data_abc1_3d(X_final, y_file_gt, win_sz, step_sz)
        if Xw.size > 0: X_all_win.append(Xw); y_all_win.append(yw)

    if not X_all_win:
        n_feat = getattr(pca_model, 'n_components_', X_file.shape[1] if 'X_file' in locals() else 1)
        return np.array([]).reshape(0, win_sz, n_feat if n_feat > 0 else 1), np.array([])
    return np.concatenate(X_all_win, axis=0), np.concatenate(y_all_win, axis=0)

# further splits the training files into a training subset and a validation set
# this validation set is used during model.fit to monitor performance
stratify_labels_for_val_split = [1 if "1_pers" in str(p).lower() else 0 for p in train_files_abc1]
can_stratify_val = len(np.unique(stratify_labels_for_val_split)) > 1 and \
                   all(np.sum(np.array(stratify_labels_for_val_split) == lbl) >= 2
                       for lbl in np.unique(stratify_labels_for_val_split))
val_split_size = 0.15

if len(train_files_abc1) * val_split_size < 1 or not can_stratify_val :
    print("warning: train set too small or lacks diversity for stratified validation split. using all train files for training, no validation set from this split.")
    train_sub_files_abc1 = train_files_abc1
    val_files_abc1 = []
else:
    try:
        train_sub_files_abc1, val_files_abc1 = train_test_split(
            train_files_abc1, test_size=val_split_size,
            stratify=stratify_labels_for_val_split, random_state=SEED, shuffle=True
        )
    except ValueError:
        print("warning: stratified validation split failed. using random split.")
        train_sub_files_abc1, val_files_abc1 = train_test_split(train_files_abc1, test_size=val_split_size, random_state=SEED, shuffle=True)


print(f"abc1 final training files: {len(train_sub_files_abc1)}")
print(f"abc1 final validation files: {len(val_files_abc1)}")

# process the train, validation, and test file sets separately
X_train_abc1, y_train_abc1 = process_data_split_abc1(
    train_sub_files_abc1, scaler_loaded_abc1, pca_loaded_abc1,
    WIN_SIZE_ABC1, STEP_SIZE_ABC1, apply_dwt_after_pca_flag_abc1, DWT_BEFORE_PCA
)
if val_files_abc1:
    X_val_abc1, y_val_abc1 = process_data_split_abc1(
        val_files_abc1, scaler_loaded_abc1, pca_loaded_abc1,
        WIN_SIZE_ABC1, STEP_SIZE_ABC1, apply_dwt_after_pca_flag_abc1, DWT_BEFORE_PCA
    )
else:
    n_feat_val = getattr(pca_loaded_abc1, 'n_components_', X_stack_abc1.shape[1] if 'X_stack_abc1' in locals() else 1)
    X_val_abc1, y_val_abc1 = np.array([]).reshape(0, WIN_SIZE_ABC1, n_feat_val if n_feat_val > 0 else 1), np.array([])


X_test_abc1, y_test_abc1 = process_data_split_abc1(
    test_files_abc1, scaler_loaded_abc1, pca_loaded_abc1,
    WIN_SIZE_ABC1, STEP_SIZE_ABC1, apply_dwt_after_pca_flag_abc1, DWT_BEFORE_PCA
)

print(f"\nabc1 processed train data -> x:{X_train_abc1.shape}, y:{y_train_abc1.shape}")
if y_train_abc1.size > 0: print(f"  unique y_train_abc1 labels: {np.unique(y_train_abc1)}")
print(f"abc1 processed val data   -> x:{X_val_abc1.shape}, y:{y_val_abc1.shape}")
if y_val_abc1.size > 0: print(f"  unique y_val_abc1 labels: {np.unique(y_val_abc1)}")
print(f"abc1 processed test data  -> x:{X_test_abc1.shape}, y:{y_test_abc1.shape}")
if y_test_abc1.size > 0: print(f"  unique y_test_abc1 labels: {np.unique(y_test_abc1)}")

# saves the final windowed numpy arrays to disk
np.save(ABC1_TRAIN_WINDOW_DIR / "X_train_windows.npy", X_train_abc1)
np.save(ABC1_TRAIN_WINDOW_DIR / "y_train_labels.npy", y_train_abc1)
np.save(ABC1_VAL_WINDOW_DIR / "X_val_windows.npy", X_val_abc1)
np.save(ABC1_VAL_WINDOW_DIR / "y_val_labels.npy", y_val_abc1)
np.save(ABC1_TEST_WINDOW_DIR / "X_test_windows.npy", X_test_abc1)
np.save(ABC1_TEST_WINDOW_DIR / "y_test_labels.npy", y_test_abc1)
print(f"final windowed data for {EXPERIMENT_NAME} saved to {ABC1_WINDOWED_DATA_DIR}")

print(f"dwt was applied {'before' if DWT_BEFORE_PCA else 'after'} pca for this abc1 run.")
print("\n✔ cell 5: windowing complete for abc1.ipynb.")

In [ ]:
# %%
# abc1.ipynb - cell 6: mlp training and evaluation

print(f"\n--- starting mlp training & evaluation for: {EXPERIMENT_NAME} ---")

if 'X_train_abc1' not in locals() or X_train_abc1.size == 0 or \
   'X_val_abc1' not in locals() or \
   'X_test_abc1' not in locals() or X_test_abc1.size == 0:
    print("error: training, validation, or test data for abc1 is empty or not defined. skipping mlp training.")
else:
    # flattens the 3d windowed data (samples, timesteps, features) into 2d for the mlp
    def flatten_abc1(X_3d: np.ndarray) -> np.ndarray:
        if X_3d.ndim == 3:
            n_s, n_t, n_f = X_3d.shape
            if n_f == 0 and n_s > 0: return np.array([]).reshape(n_s, 0)
            return X_3d.reshape(n_s, n_t * n_f).astype(np.float32)
        return X_3d.astype(np.float32)

    X_tr_flat_abc1 = flatten_abc1(X_train_abc1)

    if X_val_abc1.size > 0:
        X_va_flat_abc1 = flatten_abc1(X_val_abc1)
    else:
        X_va_flat_abc1 = np.array([]).reshape(0, X_tr_flat_abc1.shape[1] if X_tr_flat_abc1.size > 0 else 0)

    X_test_flat_abc1 = flatten_abc1(X_test_abc1)

    if X_tr_flat_abc1.shape[0] > 0 and X_tr_flat_abc1.shape[1] == 0 and X_train_abc1.shape[0] > 0 and X_train_abc1.shape[2] > 0 :
        print("error: input dimension for mlp is 0 after flattening (likely no pca components). cannot proceed.")
    elif X_tr_flat_abc1.shape[0] == 0 :
        print("error: training data for abc1 is empty after flattening. cannot proceed.")
    else:
        input_dim_abc1 = X_tr_flat_abc1.shape[1]
        print(f"abc1 mlp input dimension: {input_dim_abc1}")

        # calculates class weights to counteract any imbalance in the training data
        cw_dict_abc1 = None
        if len(y_train_abc1) > 0 and len(np.unique(y_train_abc1)) > 1:
            try:
                cw_vals_abc1 = class_weight.compute_class_weight(class_weight="balanced", classes=np.unique(y_train_abc1), y=y_train_abc1)
                cw_dict_abc1 = {int(c): float(w) for c, w in zip(np.unique(y_train_abc1), cw_vals_abc1)}
            except ValueError: cw_dict_abc1 = None
        print(f"abc1 class weights: {cw_dict_abc1}")

        # defines the mlp architecture, matching table ii in the paper
        def build_mlp_abc1(dim, name_suffix=""):
            inp = layers.Input(shape=(dim,))
            x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4))(inp)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(0.3)(x)
            x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(0.3)(x)
            x = layers.Dense(64,  kernel_regularizer=regularizers.l2(1e-4))(x)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(0.2)(x)
            x = layers.Dense(32,  kernel_regularizer=regularizers.l2(1e-4))(x)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
            out = layers.Dense(1, activation="sigmoid")(x)
            model = tf.keras.Model(inp, out, name=f"MLP_ABC1_{EXPERIMENT_SUFFIX}{name_suffix}")
            model.compile(optimizer=tf.keras.optimizers.Adam(MLP_LEARNING_RATE_ABC1), loss="binary_crossentropy", metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")])
            return model

        model_suffix_detail = f"_W{WIN_SIZE_ABC1}_S{STEP_SIZE_ABC1}_DWT_{'BeforePCA' if DWT_BEFORE_PCA else 'AfterPCA'}"
        mlp_abc1_model_name = f"MLP_ABC1{model_suffix_detail}"

        mlp_abc1 = build_mlp_abc1(input_dim_abc1, name_suffix=model_suffix_detail)
        print(f"\n--- training {mlp_abc1_model_name} for {EXPERIMENT_NAME} ---")
        mlp_abc1.summary()

        model_checkpoint_path_abc1 = str(ABC1_MODELS_DIR / mlp_abc1_model_name / "best_model.keras")
        (ABC1_MODELS_DIR / mlp_abc1_model_name).mkdir(parents=True, exist_ok=True)

        validation_data_fit = None
        if X_va_flat_abc1.size > 0 and y_val_abc1.size > 0:
            validation_data_fit = (X_va_flat_abc1, y_val_abc1.astype(np.int32))
        else:
            print("no validation data available for this training run (x_va_flat_abc1 or y_val_abc1 is empty).")

        # trains the model using the specified parameters and callbacks
        # callbacks for learning rate reduction and early stopping prevent overfitting
        history_abc1 = mlp_abc1.fit(
            X_tr_flat_abc1, y_train_abc1.astype(np.int32),
            epochs=MLP_EPOCHS_ABC1, batch_size=MLP_BATCH_SIZE_ABC1,
            validation_data=validation_data_fit,
            class_weight=cw_dict_abc1,
            callbacks=[
                callbacks.ReduceLROnPlateau(monitor='val_loss' if validation_data_fit else 'loss', patience=10, factor=0.5, min_lr=1e-6, verbose=1),
                callbacks.EarlyStopping(monitor='val_loss' if validation_data_fit else 'loss', patience=15, restore_best_weights=True, verbose=1),
                callbacks.ModelCheckpoint(filepath=model_checkpoint_path_abc1, save_best_only=True,
                                          monitor='val_accuracy' if validation_data_fit else 'accuracy',
                                          mode='max', verbose=1)
            ], verbose=2
        )

        if Path(model_checkpoint_path_abc1).exists(): mlp_abc1.load_weights(model_checkpoint_path_abc1)

        # evaluates the best saved model on the unseen test set
        print(f"\n--- evaluating {mlp_abc1_model_name} on abc1 test set ---")
        eval_results_abc1 = mlp_abc1.evaluate(X_test_flat_abc1, y_test_abc1.astype(np.int32), verbose=0)
        y_prob_abc1 = mlp_abc1.predict(X_test_flat_abc1).ravel()
        y_pred_abc1 = (y_prob_abc1 > 0.5).astype(int)

        # prints all relevant performance metrics
        print(f"\n{mlp_abc1_model_name} - test results ({EXPERIMENT_NAME}):")
        print(f"  loss: {eval_results_abc1[0]:.4f}")
        print(f"  accuracy: {eval_results_abc1[1]:.4f}")
        print(f"  precision (keras): {eval_results_abc1[2]:.4f}")
        print(f"  recall (keras): {eval_results_abc1[3]:.4f}")
        f1_abc1 = f1_score(y_test_abc1, y_pred_abc1, pos_label=1, average='binary', zero_division=0)
        prec_sk_abc1 = precision_score(y_test_abc1, y_pred_abc1, pos_label=1, average='binary', zero_division=0)
        rec_sk_abc1 = recall_score(y_test_abc1, y_pred_abc1, pos_label=1, average='binary', zero_division=0)
        print(f"  f1-score (sklearn class 1): {f1_abc1:.4f}")
        print(f"  precision (sklearn class 1): {prec_sk_abc1:.4f}")
        print(f"  recall (sklearn class 1): {rec_sk_abc1:.4f}")
        roc_auc_abc1 = float('nan')
        if len(np.unique(y_test_abc1)) > 1:
            try: roc_auc_abc1 = roc_auc_score(y_test_abc1, y_prob_abc1)
            except ValueError: pass
        if np.isnan(roc_auc_abc1): print("  roc auc: could not be calculated")
        else: print(f"  roc auc: {roc_auc_abc1:.4f}")

        mlp_abc1.save(ABC1_MODELS_DIR / f"{mlp_abc1_model_name}_final.keras")
        print(f"✔ {mlp_abc1_model_name} (final) saved to {ABC1_MODELS_DIR}.")

        # generates and saves evaluation plots
        abc1_plot_save_dir = ABC1_PLOTS_DIR / mlp_abc1_model_name
        abc1_plot_save_dir.mkdir(parents=True, exist_ok=True)

        # --- local plotting functions for abc1.ipynb ---
        def plot_training_history_abc1(history, model_name_str, save_dir):
            plt.figure(figsize=(14, 6)); plt.subplot(1, 2, 1)
            if 'accuracy' in history.history: plt.plot(history.history['accuracy'], label='train acc', marker='.')
            if 'val_accuracy' in history.history: plt.plot(history.history['val_accuracy'], label='val acc', marker='.')
            plt.title('accuracy', fontsize=14); plt.xlabel('epoch'); plt.ylabel('accuracy')
            if 'accuracy' in history.history or 'val_accuracy' in history.history: plt.legend()
            plt.grid(True, alpha=0.7); plt.subplot(1, 2, 2)
            if 'loss' in history.history: plt.plot(history.history['loss'], label='train loss', marker='.')
            if 'val_loss' in history.history: plt.plot(history.history['val_loss'], label='val loss', marker='.')
            plt.title('loss', fontsize=14); plt.xlabel('epoch'); plt.ylabel('loss')
            if 'loss' in history.history or 'val_loss' in history.history: plt.legend()
            plt.grid(True, alpha=0.7); plt.suptitle(f"training: {model_name_str}", fontsize=16, fontweight='bold')
            plt.tight_layout(rect=[0,0.03,1,0.95]); plt.savefig(save_dir/"th.png",dpi=PLOT_DPI); plt.close(); print(f"plot: {save_dir/'th.png'}")
        def plot_confusion_matrix_abc1(y_true, y_pred, model_name_str, class_names, save_dir):
            cm=confusion_matrix(y_true,y_pred); plt.figure(figsize=(7,6))
            sns.heatmap(cm,annot=True,fmt='d',cmap='blues',xticklabels=class_names,yticklabels=class_names,annot_kws={"size":16},cbar=False)
            plt.title(f'{model_name_str} - cm',fontsize=16,fontweight='bold');plt.xlabel('predicted');plt.ylabel('actual');plt.tight_layout()
            plt.savefig(save_dir/"cm.png",dpi=PLOT_DPI);plt.close(); print(f"plot: {save_dir/'cm.png'}")
        def plot_roc_curve_abc1(y_true, y_prob, model_name_str, auc, save_dir):
            fpr,tpr,_=roc_curve(y_true,y_prob);plt.figure(figsize=(8,7))
            plt.plot(fpr,tpr,label=f'auc = {auc:.4f}',color='darkorange',lw=2.5);plt.plot([0,1],[0,1],'k--',lw=2)
            plt.xlim([-0.01,1.0]);plt.ylim([0.0,1.01]);plt.xlabel('fpr');plt.ylabel('tpr')
            plt.title(f'{model_name_str} - roc',fontsize=16,fontweight='bold');plt.legend(loc="lower right");plt.grid(True,alpha=0.7);plt.tight_layout()
            plt.savefig(save_dir/"roc.png",dpi=PLOT_DPI);plt.close(); print(f"plot: {save_dir/'roc.png'}")
        def plot_pr_curve_abc1(y_true, y_prob, model_name_str, save_dir):
            prec,rec,_=precision_recall_curve(y_true,y_prob);plt.figure(figsize=(8,7))
            plt.plot(rec,prec,label='pr curve',color='mediumblue',lw=2.5)
            plt.xlabel('recall');plt.ylabel('precision');plt.ylim([0.0,1.01]);plt.xlim([0.0,1.0])
            plt.title(f'{model_name_str} - pr curve',fontsize=16,fontweight='bold');plt.grid(True,alpha=0.7);plt.tight_layout()
            plt.savefig(save_dir/"pr_curve.png",dpi=PLOT_DPI);plt.close(); print(f"plot: {save_dir/'pr_curve.png'}")

        plot_training_history_abc1(history_abc1, mlp_abc1_model_name, abc1_plot_save_dir)
        plot_confusion_matrix_abc1(y_test_abc1, y_pred_abc1, mlp_abc1_model_name, ["no person (0)", "person (1)"], abc1_plot_save_dir)
        if not np.isnan(roc_auc_abc1): plot_roc_curve_abc1(y_test_abc1, y_prob_abc1, mlp_abc1_model_name, roc_auc_abc1, abc1_plot_save_dir)
        plot_pr_curve_abc1(y_test_abc1, y_prob_abc1, mlp_abc1_model_name, abc1_plot_save_dir)

        print(f"\n✔ cell 6: {mlp_abc1_model_name} training and evaluation complete for {EXPERIMENT_NAME}.")